# 🏠 Real Estate RFM Lead Segmentation — Walkthrough Notebook

Εκπαιδευτικό Jupyter Notebook για το μάθημα **Data Analyst**:
1. Φόρτωση CRM δεδομένων (`sample_crm_leads.csv` & `sample_interactions.csv`).
2. Έλεγχος γιατί **πρέπει** να φιλτράρουμε `direction == 'inbound'` (αποφυγή data leakage από δικά μας follow-ups).
3. Εφαρμογή των 5 segments (`Έτοιμος αγοραστής`, `Νέος ενδιαφερόμενος`, `Σε σκέψη`, `Σε ψύξη`, `Σε νάρκη`).
4. Εντοπισμός πελατών σε **Alert 45–60 ημερών** & Marketing Attribution ανά περιοχή/κανάλι.

In [ ]:
import pandas as pd

leads = pd.read_csv('../data/sample_crm_leads.csv')
interactions = pd.read_csv('../data/sample_interactions.csv')

print(f'Πελάτες: {len(leads)} | Συνολικές επικοινωνίες: {len(interactions)}')
leads.head()

## Βήμα 1: Inbound vs Outbound — Γιατί έχει τεράστια σημασία
Δες πόσες επικοινωνίες είναι δικές μας (`outbound`) vs του πελάτη (`inbound`). Αν δεν φιλτράραμε `inbound`, ανενεργοί πελάτες θα φαίνονταν «φρέσκοι»!

In [ ]:
interactions['direction'].value_counts()

## Βήμα 2: Κατηγοριοποίηση στα 5 Segments + Alert 45–60 ημερών

In [ ]:
def classify(row):
    days = row['days_since_inbound']
    signals = row['signals_count']
    visit = bool(row['has_visit'])
    budget = row['budget']

    if days <= 60:
        if visit or signals >= 4 or (2 <= signals <= 3 and budget >= 200_000):
            return 'Έτοιμος αγοραστής'
        return 'Νέος ενδιαφερόμενος'
    if days <= 180:
        return 'Σε σκέψη'
    if days <= 365:
        return 'Σε ψύξη'
    return 'Σε νάρκη'

leads['segment'] = leads.apply(classify, axis=1)
leads['at_risk_60d'] = (leads['segment'] == 'Έτοιμος αγοραστής') & (leads['days_since_inbound'].between(45, 60))

summary = (
    leads.groupby('segment')
    .agg(
        leads_count=('lead_id', 'count'),
        total_budget=('budget', 'sum'),
        avg_budget=('budget', 'mean'),
        at_risk_alerts=('at_risk_60d', 'sum'),
    )
    .round(0)
)
summary

## Βήμα 3: Marketing Attribution — Ποια κανάλια φέρνουν τους περισσότερους «Έτοιμους αγοραστές»;

In [ ]:
leads['is_ready'] = leads['segment'] == 'Έτοιμος αγοραστής'
channel_perf = (
    leads.groupby('source')
    .agg(total_leads=('lead_id', 'count'), ready_leads=('is_ready', 'sum'))
    .assign(ready_rate_pct=lambda d: (d['ready_leads'] / d['total_leads'] * 100).round(1))
    .sort_values('ready_rate_pct', ascending=False)
)
channel_perf